# Cognopolis · Урок M6.5 — Подземье: агент-рудокоп (домашка)

В воркспейсе день шахтёра собирали за вас — теперь соберите его сами. Каркас уже рабочий:
`Run all` проходит целиком **и до, и после** ваших правок. Ваша работа — три политики
(`# TODO`) и задача со звёздочкой.

> **Working-first.** Нужен только токен своего аккаунта (секрет `COGNOPOLIS_TOKEN`).
> LLM-ключ и золото не нужны. Демо-аккаунт `testuser` в шахту не пускает демо-гард.
> Помните боль Подземья: пять промахов на серию, промах = минус hp, смерть жжёт вьюк
> целиком (рюкзак и склад целы, житель у входа с hp 1). С данным решателем вы не
> промахнётесь — но политики пишете уже вы.

**Ссылки:** лекция — сайт курса, «Подземье: агент-рудокоп (M6.5)» ·
API-доки: `https://kindomklaster.com/docs` · сначала руками: тайл входа → «Спуститься
в шахту» → панель «Копать» на жиле (там же блок «что понадобится агенту»).

## 1. Сетап

In [ ]:
%pip install -q "cognopolis-client @ git+https://github.com/ITrubnikov/Train_of_Thought-Cognopolis-client.git"

In [ ]:
import os, time
from cognopolis_client import Client, GameError

# ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Свой сервер — через переменную COGNOPOLIS_URL.
BASE_URL = os.environ.get("COGNOPOLIS_URL", "https://kindomklaster.com")

def get_secret(name: str) -> str:
    """Секрет из env / Colab / Kaggle — одним хелпером (канон курса)."""
    val = os.environ.get(name, "")
    if val:
        return val
    try:
        from google.colab import userdata            # Colab: значок ключа слева -> Secrets
        return userdata.get(name) or ""
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle: Add-ons -> Secrets
        return UserSecretsClient().get_secret(name) or ""
    except Exception:
        pass
    return ""

# Токен жителя: Ратуша -> вкладка «аккаунт» -> «копировать». Демо-аккаунт testuser в шахту
# не пускает демо-гард (demo_forbidden) — нужен свой аккаунт.
TOKEN = get_secret("COGNOPOLIS_TOKEN")
assert TOKEN, (f"Нужен токен жителя мира {BASE_URL} — секрет/переменная COGNOPOLIS_TOKEN "
               "(Ратуша -> вкладка «аккаунт»).")

c = Client(BASE_URL, token=TOKEN)

# Мягкая проверка связи: если мир недоступен — живые ячейки ниже честно пропустятся.
WORLD_UP = True
try:
    c.get_map()                       # GET /map не требует токена
except Exception as e:
    WORLD_UP = False
    print(f"Мир {BASE_URL} недоступен ({type(e).__name__}). Живые ячейки пропущу — "
          "проверь COGNOPOLIS_URL или попробуй позже.")

print("Мир:", BASE_URL, "| на связи:", WORLD_UP)
print("Смотреть за агентом в браузере:", f"{BASE_URL}/?token={TOKEN}")

## 2. Разогрев — карта мира, карта горизонта, навигация

Вход — тайл `mine_entrance` на карте мира; карта горизонта публична и статична. Навигация
под землёй — те же 8 направлений, что в M1; меняется только конверт (`miner` вместо
`character`).

In [ ]:
if WORLD_UP:
    ch = c.get_character()
    print(f"Житель: ({ch['x']},{ch['y']}), hp {ch['hp']}, под землёй: {ch.get('in_mine', False)}")

    # Вход в Подземье — обычный тайл на карте мира; ищем его, а не хардкодим.
    world = c.get_map()
    ENTRANCE = next((t["x"], t["y"]) for t in world["tiles"]
                    if t.get("content") == "mine_entrance")
    print("Вход в Подземье на карте мира:", ENTRANCE)

    # Карта горизонта 1 — публичная и статичная (меняется только состояние жил): кэшируй смело.
    mm = c.mine_map(horizon=1)
    GLYPH = {"entry": "E", "descent": "D", "vein": "\u2116", "empty": "\u00b7"}
    grid = [["\u00b7"] * mm["size"] for _ in range(mm["size"])]
    for t in mm["tiles"]:
        grid[t["y"]][t["x"]] = GLYPH.get(t["content"], "?")
    print("\nГоризонт 1 (E — вход/выход, \u2116 — жила, D — ствол глубже):")
    for row in grid:
        print("  " + " ".join(row))

    VEINS = [(t["x"], t["y"]) for t in mm["tiles"] if t["content"] == "vein"]
    MINE_ENTRY = next((t["x"], t["y"]) for t in mm["tiles"] if t["content"] == "entry")
    print(f"\nЖилы горизонта 1: {VEINS}; entry (вход = выход): {MINE_ENTRY}")
else:
    print("мир недоступен — пропуск")

In [ ]:
# 8 направлений — как в M1 (D-069); диагонали экономят ходы. Так ходят И наверху, И в шахте:
# навигация — общая механика, «вообще по-другому» под землёй работает только добыча.
STEP_DIR = {(1, 0): "east", (-1, 0): "west", (0, 1): "south", (0, -1): "north",
            (1, 1): "southeast", (-1, 1): "southwest",
            (1, -1): "northeast", (-1, -1): "northwest"}

def sign(v: int) -> int:
    return (v > 0) - (v < 0)

def walk_to(x: int, y: int, why: str = "") -> dict:
    """Шагать по ПОВЕРХНОСТИ до (x, y). Конверт игры: {result, cooldown, character}."""
    ch = c.get_character()
    while (ch["x"], ch["y"]) != (x, y):
        step = (sign(x - ch["x"]), sign(y - ch["y"]))
        r = c.move_dir(STEP_DIR[step], reason=why)
        ch = r["character"]
        if r["result"].get("banked"):
            print("  дом: авто-банк", r["result"]["banked"])
        c.wait_cooldown()
    return ch

def mine_walk_to(x: int, y: int, why: str = "") -> dict:
    """Шагать по ГОРИЗОНТУ ШАХТЫ до (x, y). Конверт сервиса: {result, cooldown, miner}."""
    m = c.mine_observe()
    while (m["x"], m["y"]) != (x, y):
        step = (sign(x - m["x"]), sign(y - m["y"]))
        r = c.mine_move(STEP_DIR[step], reason=why)
        m = r["miner"]
        c.wait_cooldown()
    return m

print("Навигация готова: walk_to (поверхность) и mine_walk_to (шахта).")

## 3. Разбор — Испытание как данные, решатель как Приём

Добычу в шахте гейтит не кулдаун, а задача: `mine_trial()` даёт вопрос и файлы,
`mine_attempt(answer, series=...)` принимает ответ. Решатель горизонта 1 дан целиком —
это легально: первый горизонт бесплатный, границу сложности держат следующие. Ваш рост —
в политиках вокруг него.

In [ ]:
import csv, io, re

# Правила датасета горизонта 1: записка называет товар в «ёлочках» (единственная такая пара
# в тексте) и месяц — одним словом с заглавной после слова «месяц».
GOOD_RE = re.compile(r"«([^»]+)»")
MONTH_RE = re.compile(r"месяц\s+([А-ЯЁ][а-яё]+)")

def solve(files: dict[str, bytes]) -> str:
    """Приём горизонта 1: «счётные печати». Читает записку, фильтрует гроссбух по товару
    И месяцу, возвращает сумму количество*цена строкой — ровно то, что спрашивает печать."""
    note = files["note.txt"].decode("utf-8")
    good = GOOD_RE.search(note).group(1)
    month = MONTH_RE.search(note).group(1)
    total = sum(
        int(row["количество"]) * int(row["цена"])
        for row in csv.DictReader(io.StringIO(files["ledger.csv"].decode("utf-8")))
        if row["товар"] == good and row["месяц"] == month)
    return str(total)

# Приём — это код: его можно проверить БЕЗ шахты, на своих файлах.
demo_files = {
    "note.txt": "Смотритель велит счесть выручку за «гвозди» в месяц Ковеня.".encode(),
    "ledger.csv": ("товар,месяц,количество,цена\n"
                   "гвозди,Ковеня,3,5\n"
                   "гвозди,Ливеня,10,2\n"
                   "верёвка,Ковеня,7,4\n").encode(),
}
assert solve(demo_files) == "15", "3 гвоздя по 5 в Ковене = 15"
print("Решатель жив: demo-печать даёт", solve(demo_files))

## 4. Задачи — доведите день шахтёра до умного

Каркас `mining_day()` ниже работает сразу. Три политики с рабочими дефолтами — ваши `# TODO`:

1. **Маршрут** (`choose_vein`) — ближайшая жила вместо первой попавшейся.
2. **Push-your-luck** (`should_ascend`) — когда пора наверх: hp, вьюк, жадность.
3. **Робастность** (`on_refusal` + петля) — `series_changed` не должен терять жилу.

In [ ]:
ORE_TARGET = 2      # вежливость к общему миру: пара зарядов за забег — и наверх


def choose_vein(m: dict, veins: list, visited: set):
    """К какой жиле идти. m — снимок miner (x, y, hp, satchel...), veins — все жилы
    горизонта, visited — уже обойдённые в этом забеге.

    TODO 1 (маршрут): верните БЛИЖАЙШУЮ непосещённую жилу — метрика Чебышёва
    max(|dx|, |dy|): диагональ стоит один ход, как и прямой шаг.
    Рабочий дефолт: первая непосещённая по порядку списка (работает, но ходит зигзагами)."""
    options = [v for v in veins if v not in visited]
    return options[0] if options else None


def should_ascend(m: dict, mined: int) -> bool:
    """Пора ли наверх. Вернуть True = идём к entry фиксировать вьюк.

    TODO 2 (push-your-luck): напишите СВОЮ политику. Идеи: подниматься при hp ниже порога
    (промахи бьют по жизни, а смерть жжёт вьюк); брать больше при пустой шахте; учитывать,
    сколько уже на складе. Помните: вьюк на кону, пока вы внизу.
    Рабочий дефолт: подняться, когда добыто ORE_TARGET или вьюк полон."""
    return mined >= ORE_TARGET or sum(m["satchel"].values()) >= m["satchel_cap"]


def on_refusal(code: str) -> bool:
    """Что делать с отказом сервиса в петле добычи. True = «понятный отказ, иди дальше»,
    False = неожиданная ошибка, надо поднять (raise).

    TODO 3 (робастность): сейчас series_changed тоже «иди дальше» — а ведь это просто
    смена экземпляра испытания: жилу выбили и реставрировали. Правильнее — перечитать
    mine_trial() и подать ответ заново на той же жиле. Перестройте петлю mining_day так,
    чтобы series_changed не терял жилу.
    Рабочий дефолт: все знакомые отказы шахты — «иди дальше»."""
    return code in {"vein_depleted", "no_vein_here", "series_changed",
                    "attempt_budget_exhausted", "satchel_full", "character_on_cooldown"}


print("Политики дня шахтёра готовы (дефолты рабочие — улучшайте по одной).")

In [ ]:
def mining_day() -> dict:
    """День шахтёра целиком: спуск -> жилы -> подъём -> зачисление -> домой.
    Каркас рабочий как есть; ваши политики choose_vein/should_ascend/on_refusal он
    подхватывает автоматически."""
    ch = c.get_character()
    if not ch.get("in_mine"):
        walk_to(*ENTRANCE, why="к входу в шахту")
        try:
            print("спуск:", c.mine_descend(reason="день шахтёра")["result"])
            c.wait_cooldown()
        except GameError as e:
            if e.code != "in_mine":
                raise
            print("уже под землёй — продолжаем забег")
    else:
        print("уже под землёй — продолжаем забег")

    m = c.mine_observe()
    mined, visited = sum(m["satchel"].values()), set()
    while not should_ascend(m, mined):
        vein = choose_vein(m, VEINS, visited)
        if vein is None:
            print("непосещённых жил не осталось — наверх")
            break
        visited.add(vein)
        m = mine_walk_to(*vein, why="к жиле")
        while not should_ascend(m, mined):
            try:
                trial = c.mine_trial()                   # вопрос + артефакты (бесплатно)
                answer = solve(c.mine_trial_files())     # ваш Приём
                r = c.mine_attempt(answer, series=trial["progress"]["series"])
                c.wait_cooldown()
            except GameError as e:
                if not on_refusal(e.code):
                    raise
                print(f"  жила {vein}: отказ {e.code} — дальше")
                time.sleep(1)
                break
            out, m = r["result"], r["miner"]
            if out["outcome"] == "hit":
                mined = sum(m["satchel"].values())
                print(f"  hit: +{out['ore']} — вьюк {m['satchel']}, hp {m['hp']}")
                if out.get("vein") == "depleted":
                    break
            else:
                print(f"  miss: hp={out['hp']}, бюджет={out['budget_left']} — проверь решатель!")
                break

    try:
        mine_walk_to(*MINE_ENTRY, why="к стволу — фиксировать вьюк")
        print("финализация:", c.mine_up(reason="подъём: фиксирую вьюк")["result"])
        c.wait_cooldown()
    except GameError as e:
        if e.code not in {"invalid_token", "run_not_active"}:
            raise
        print("забег уже финализирован — к зачислению")
    try:
        print("зачисление:", c.mine_return(reason="за рудой")["result"])
        c.wait_cooldown()
    except GameError as e:
        if e.code not in {"not_in_mine", "nothing_to_settle"}:
            raise
        print(f"зачислять нечего ({e.code})")
    return walk_to(0, 0, why="домой — сдать медь")


if WORLD_UP:
    T_DAY = time.time()
    ch = mining_day()
    print(f"\nДень занял {time.time() - T_DAY:.0f}с. "
          f"Рюкзак: {ch['inventory']} | склад copper: {ch['stored'].get('copper', 0)}")
else:
    print("мир недоступен — пропуск")

## 5. Проверка

Критерий приёма: на складе лежит хотя бы одна медь, а в логе виден полный цикл
спуск → hit → финализация → зачисление → авто-банк. Пороги — «не меньше»: мир общий.

In [ ]:
if WORLD_UP:
    ch = c.get_character()
    copper = ch["stored"].get("copper", 0)
    assert copper >= 1, (
        "На складе нет меди. Сплошные vein_depleted в логе? Жилы выбиты другими шахтёрами — "
        "подожди реставрации (минуты) и перезапусти ячейку с mining_day().")
    print(f"Медь на складе: {copper} — критерий урока выполнен.")
    print("Дальше — улучшайте политики: TODO 1 (маршрут), TODO 2 (push-your-luck), "
          "TODO 3 (series_changed без потери жилы).")
else:
    print("мир недоступен — пропуск проверки")

## ★ Со звёздочкой — тесты Приёма без сети

Смерть отбирает вьюк, но не Приём. Значит, Приём можно (и нужно) укреплять как обычный
код — юнит-тестами на локальных файлах, без единого запроса к шахте.

In [ ]:
# ★ Задача со звёздочкой: Приём — это код, тестируйте его без шахты.
# Соберите свои наборы файлов (краевые случаи!) и погоняйте solve() локально.
star_cases = [
    # (files, ожидаемый ответ)
    ({"note.txt": "Счесть за «уголь» в месяц Стуженя.".encode(),
      "ledger.csv": ("товар,месяц,количество,цена\n"
                     "уголь,Стуженя,2,10\n"
                     "уголь,Ковеня,9,9\n").encode()}, "20"),
    # TODO ★: добавьте случаи — товар встречается в нескольких месяцах; нулевое количество;
    # товар с пробелом в имени («железные скобы»); месяц упомянут в записке дважды.
]
for files, expected in star_cases:
    got = solve(files)
    assert got == expected, f"ожидали {expected}, получили {got}"
print(f"Локальные тесты Приёма зелёные: {len(star_cases)} случая(ев).")

## Наблюдаемость — смотрите за своим рудокопом

`BASE_URL/?token=<ваш токен>`: тумблер «карта/шахта», журнал забега (каждый hit/miss),
вехи в Хронике («спустился в шахту», «вышел с грузом»). Пока агент читает файлы и считает,
интерфейс честно пишет «Решает печать…» — длинная decide-фаза в Подземье легальна.

Артефакт ДЗ — публичная ссылка на прогнанный ноутбук в чат курса: `[Урок M6.5, ДЗ] {ссылка}`.